# 19. Vector Databases Integration (serverless FAISS)

A **vector database** stores your text as **numbers (vectors)** so an agent can **search by meaning**.

**FAISS** is a free, **serverless** vector database — "serverless" means it runs **on your own computer**, with no server to set up and no cloud cost.

## Why a vector database?

When you have **lots** of text, you can't give it all to the agent at once. A vector database lets the agent **fetch only the most relevant pieces** for each question.

### Real-life analogy
It's a **smart filing cabinet** that hands you only the few pages you need — sorted by meaning, not by exact words.

## Install (free, local)
```
pip install faiss-cpu langchain-community langchain-huggingface sentence-transformers
```
No cloud account needed — everything runs on your machine.

## Step 1: Build a small FAISS database

We turn a few sentences into a searchable FAISS store using **free local embeddings** (no API key).

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

facts = [
    "BlueShoe was founded in 2019.",
    "We sell eco-friendly shoes made from recycled plastic.",
    "Our shoes come in red, blue, and green.",
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
faiss_db = FAISS.from_texts(facts, embeddings)
print("FAISS database built (running locally)!")

c:\Users\THIRU\OneDrive\Desktop\gen_ai_course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\THIRU\AppData\Local\Temp\ipykernel_42064\2305488318.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3589.82it/s]


FAISS database built (running locally)!


## Step 2: Search it by meaning

We ask in our own words; FAISS returns the closest facts.

In [2]:
results = faiss_db.similarity_search("What colors are available?", k=1)
print("Best match:", results[0].page_content)

Best match: Our shoes come in red, blue, and green.


## Step 3: Wrap it as a tool for an agent

We make a tiny custom tool that searches FAISS, then give it to an agent.

In [3]:
from dotenv import load_dotenv
load_dotenv()

from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

@tool("FAISS Search")
def search_facts(question: str) -> str:
    """Search the company facts and return the best matching fact."""
    hits = faiss_db.similarity_search(question, k=1)
    return hits[0].page_content

llm = LLM(model="gpt-4o-mini")
agent = Agent(
    role="Company Helper",
    goal="Answer using the FAISS search tool",
    backstory="You always search the facts before answering.",
    tools=[search_facts],
    llm=llm,
)

task = Task(
    description="What colors do our shoes come in?",
    expected_output="The colors.",
    agent=agent,
)

crew = Crew(agents=[agent], tasks=[task])

# In a Jupyter notebook, use kickoff_async() with "await".
result = await crew.kickoff_async()
print(result)

Our shoes come in red, blue, and green.


## Key points to remember

- A **vector database** stores text as numbers so you can **search by meaning**.
- **FAISS** is free and **serverless** — it runs on your own computer.
- Build it with `FAISS.from_texts(texts, embeddings)` using free local embeddings.
- Search with `similarity_search(question, k=...)`.
- Wrap the search in a **custom tool** so a CrewAI agent can use it.